In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Estimare de utilizare: 7 minute pe un procesor Heron r2 (NOTĂ: Aceasta este doar o estimare. Timpul tău de execuție poate varia.)*

##  Rezultate ale învățării

Sugerăm ca utilizatorii să fie familiarizați cu următoarele subiecte înainte de a parcurge acest tutorial:
- Noțiunile de bază ale decuplării dinamice, atenuării erorilor de măsurare, twirling-ului de poartă-uri și extrapolării la zgomot zero, descrise în acest [ghid](/guides/error-mitigation-and-suppression-techniques).

## Cerințe prealabile

După parcurgerea acestui tutorial, utilizatorii ar trebui să înțeleagă:

- Cum sunt implementate selectiv pe hardware tehnicile de atenuare a erorilor menționate anterior.
- Cum se compară în ceea ce privește capacitatea lor de a atenua zgomotul hardware.

## Background

Acest tutorial explorează opțiunile de suprimare și atenuare a erorilor disponibile cu primitiva Estimator din Qiskit Runtime. Acest tutorial arată cum să implementezi individual fiecare dintre următoarele metode:

- Decuplare dinamică
- Atenuarea erorilor de măsurare
- Twirling de poartă-uri
- Extrapolarea la zgomot zero (ZNE)

Rețineți că o alternativă la implementarea individuală a acestor tehnici este să le implementezi folosind un [nivel de reziliență](/guides/estimator-noise-management), unde `resilience_level` ia valorile 0, 1, 2:

- 0 : Nu se implementează nicio mitigare.
- 1 : Se implementează atenuarea erorilor de măsurare.
- 2 : Se implementează twirling-ul de poartă-uri, atenuarea erorilor de măsurare și ZNE.

În acest tutorial, vei construi un circuit și un observabil și vei trimite joburi folosind primitiva Estimator cu diferite combinații de setări de atenuare a erorilor. Apoi, vei reprezenta grafic rezultatele pentru a observa efectele diferitelor setări. Cea mai mare parte a tutorialului folosește un circuit de 10 qubiți pentru a facilita vizualizările, iar la final vei scala fluxul de lucru la 50 de qubiți.

## Cerințe

Înainte de a începe acest ghid, asigură-te că ai instalate următoarele:

- Qiskit SDK v2.1 sau mai recent, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.40 sau mai recent (`pip install qiskit-ibm-runtime`)

## Configurare

In [7]:
import matplotlib.pyplot as plt
import numpy as np

from qiskit.circuit.library import efficient_su2, unitary_overlap
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Batch, EstimatorV2 as Estimator

## Exemplu la scară mică pe simulator
Vom sări peste acest pas deoarece atenuarea erorilor la runtime nu este suportată pe simulatoare.
## Exemplu pe hardware
### Pasul 1: Mapează intrările clasice la o problemă cuantică
Acest ghid presupune că problema clasică a fost deja mapată la cuantic. Începe prin a construi un circuit și un observabil de măsurat. Deși tehnicile folosite aici se aplică la multe tipuri diferite de circuite, pentru simplitate acest ghid folosește circuitul [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) inclus în biblioteca de circuite Qiskit.

`efficient_su2` este un circuit cuantic parametrizat proiectat să fie executat eficient pe hardware cuantic cu conectivitate limitată între qubiți, fiind totuși suficient de expresiv pentru a rezolva probleme în domenii de aplicație precum optimizarea și chimia. Este construit prin alternarea straturilor de poartă-uri cu un singur qubit parametrizate cu un strat ce conține un model fix de poartă-uri cu doi qubiți, pentru un număr ales de repetiții. Modelul de poartă-uri cu doi qubiți poate fi specificat de utilizator. Aici poți folosi modelul integrat `pairwise` deoarece minimizează adâncimea circuitului prin împachetarea cât mai densă a Gate-urilor cu doi qubiți. Acest model poate fi executat folosind doar conectivitate liniară între qubiți.

In [8]:
n_qubits = 10
reps = 1

circuit = efficient_su2(n_qubits, entanglement="pairwise", reps=reps)

circuit.decompose().draw("mpl", scale=0.7)

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/24abd7ba-bbb8-443b-9e81-866795d39a6c-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/24abd7ba-bbb8-443b-9e81-866795d39a6c-0.avif)

Ca observabil al nostru, să luăm operatorul Pauli $Z$ care acționează pe ultimul qubit, $Z I \cdots I$.
Rețineți că faptul că ultimul qubit corespunde primului element al acestui șir se datorează utilizării de către Qiskit a notației little-endian.

In [9]:
# Z on the last qubit (index -1) with coefficient 1.0
observable = SparsePauliOp.from_sparse_list(
    [("Z", [-1], 1.0)], num_qubits=n_qubits
)

În acest moment, ai putea proceda la rularea circuitului tău și la măsurarea observabilului. Totuși, vrei și să compari rezultatul dispozitivului cuantic cu răspunsul corect — adică valoarea teoretică a observabilului, dacă circuitul ar fi fost executat fără erori. Pentru circuite cuantice mici poți calcula această valoare simulând circuitul pe un calculator clasic, dar acest lucru nu este posibil pentru circuite mai mari, la scară de utilitate. Poți ocoli această problemă cu tehnica „circuitului oglindă" (cunoscută și ca „compute-uncompute"), care este utilă pentru evaluarea performanței dispozitivelor cuantice.

#### Circuit oglindă
În tehnica circuitului oglindă, concatenezi circuitul cu circuitul său invers, care este format prin inversarea fiecărui Gate din circuit în ordine inversă. Circuitul rezultat implementează operatorul identitate, care poate fi simulat trivial. Deoarece structura circuitului original este păstrată în circuitul oglindă, executarea acestuia din urmă oferă totuși o idee despre cum s-ar descurca dispozitivul cuantic pe circuitul original.

Următoarea celulă de cod atribuie parametri aleatori circuitului tău, apoi construiește circuitul oglindă folosind clasa [`unitary_overlap`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.unitary_overlap). Înainte de oglindirea circuitului, adaugă-i o instrucțiune [barrier](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Barrier) pentru a împiedica Transpiler-ul să fuzioneze cele două părți ale circuitului de pe fiecare parte a barierei, rezultând un circuit transpilat fără niciun Gate.

In [10]:
# Generate random parameters
rng = np.random.default_rng(1234)
params = rng.uniform(-np.pi, np.pi, size=circuit.num_parameters)

# Assign the parameters to the circuit
assigned_circuit = circuit.assign_parameters(params)

# Add a barrier to prevent circuit optimization of mirrored operators
assigned_circuit.barrier()

# Construct mirror circuit
mirror_circuit = unitary_overlap(assigned_circuit, assigned_circuit)

mirror_circuit.decompose().draw("mpl", scale=0.7)

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/4dbde811-1ba9-47a8-85a0-dcaff054ed60-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/4dbde811-1ba9-47a8-85a0-dcaff054ed60-0.avif)

### Pasul 2: Optimizează problema pentru execuția pe hardware cuantic
Trebuie să optimizezi circuitul înainte de a-l rula pe hardware. Acest proces implică câțiva pași:

- Alege un layout de qubiți care mapează qubiții virtuali ai circuitului la qubiți fizici pe hardware.
- Inserează Gate-uri de swap după cum este necesar pentru a direcționa interacțiunile dintre qubiți care nu sunt conectați.
- Traduce Gate-urile din circuit în instrucțiuni [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) care pot fi executate direct pe hardware.
- Efectuează optimizări ale circuitului pentru a minimiza adâncimea circuitului și numărul de poartă-uri.

Transpiler-ul integrat în Qiskit poate efectua toți acești pași pentru tine. Deoarece acest exemplu folosește un circuit eficient pe hardware, Transpiler-ul ar trebui să poată alege un layout de qubiți care nu necesită inserarea de poartă-uri swap pentru direcționarea interacțiunilor.

Trebuie să alegi dispozitivul hardware de folosit înainte de a-ți optimiza circuitul. Următoarea celulă de cod solicită cel mai puțin ocupat dispozitiv cu cel puțin 127 de qubiți.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

In [12]:
print(backend)

<IBMBackend('ibm_fez')>


You can transpile your circuit to your chosen backend by creating a pass manager and then running the pass manager on the circuit. An easy way to create a pass manager is to use the [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) function. See [Transpile with pass managers](/docs/guides/transpile-with-pass-managers) for a more detailed explanation of transpiling with pass managers.

In [13]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=1234
)
isa_circuit = pass_manager.run(mirror_circuit)

isa_circuit.draw("mpl", idle_wires=False, scale=0.7, fold=-1)

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/027e829a-44d3-455e-b2bf-8ce0d7e26b9b-0.avif" alt="Output of the previous code cell" />

Poți transpila circuitul la Backend-ul ales creând un pass manager și rulând apoi pass manager-ul pe circuit. O modalitate ușoară de a crea un pass manager este să folosești funcția [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager). Consultă [Transpilează cu pass managers](/guides/transpile-with-pass-managers) pentru o explicație mai detaliată a transpilării cu pass managers.

In [14]:
isa_observable = observable.apply_layout(isa_circuit.layout)

print("Original observable:")
print(observable)
print()
print("Observable with layout applied:")
print(isa_observable)

Original observable:
SparsePauliOp(['ZIIIIIIIII'],
              coeffs=[1.+0.j])

Observable with layout applied:
SparsePauliOp(['IIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII'],
              coeffs=[1.+0.j])


![Output of the previous code cell](../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/027e829a-44d3-455e-b2bf-8ce0d7e26b9b-0.avif)

Circuitul transpilat conține acum doar instrucțiuni ISA. Toate Gate-urile au fost descompuse în termeni de poartă-uri $\sqrt{X}$ și rotații $R_z$, și [Gate-uri CZ](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.CZGate).

Procesul de transpilare a mapat qubiții virtuali ai circuitului la qubiți fizici pe hardware. Informațiile despre layout-ul qubiților sunt stocate în atributul `layout` al circuitului transpilat. Observabilul a fost de asemenea definit în termeni de qubiți virtuali, deci trebuie să aplici acest layout observabilului, ceea ce poți face cu metoda [`apply_layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp#apply_layout) din `SparsePauliOp`.

In [ ]:
pub = (isa_circuit, isa_observable)

jobs = []

with Batch(backend=backend) as batch:
    estimator = Estimator(mode=batch)
    estimator.options.environment.job_tags = [
        "TUT_CEM_SS"
    ]  # add tag for this small scale job
    # Set number of shots
    estimator.options.default_shots = 100_000
    # Disable runtime compilation and error mitigation
    estimator.options.resilience_level = 0

    # Run job with no error mitigation
    job0 = estimator.run([pub])
    jobs.append(job0)

    # Add dynamical decoupling (DD)
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XpXm"
    job1 = estimator.run([pub])
    jobs.append(job1)

    # Add readout error mitigation (DD + TREX)
    estimator.options.resilience.measure_mitigation = True
    job2 = estimator.run([pub])
    jobs.append(job2)

    # Add gate twirling (DD + TREX + Gate Twirling)
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    job3 = estimator.run([pub])
    jobs.append(job3)

    # Add zero-noise extrapolation (DD + TREX + Gate Twirling + ZNE)
    estimator.options.resilience.zne_mitigation = True
    estimator.options.resilience.zne.noise_factors = (1, 3, 5)
    estimator.options.resilience.zne.extrapolator = ("exponential", "linear")
    job4 = estimator.run([pub])
    jobs.append(job4)

### Step 4: Post-process and return result in desired classical format

Finally, you can analyze the data. Here you will retrieve the job results, extract the measured expectation values from them, and plot the values, including error bars of one standard deviation.

In [16]:
# Retrieve the job results
results = [job.result() for job in jobs]

# Unpack the PUB results (there's only one PUB result in each job result)
pub_results = [result[0] for result in results]

# Unpack the expectation values and standard errors
expectation_vals = np.array(
    [float(pub_result.data.evs) for pub_result in pub_results]
)
standard_errors = np.array(
    [float(pub_result.data.stds) for pub_result in pub_results]
)

# Plot the expectation values
fig, ax = plt.subplots()
labels = ["No mitigation", "+ DD", "+ TREX", "+ Twirling", "+ ZNE"]
ax.bar(
    range(len(labels)),
    expectation_vals,
    yerr=standard_errors,
    label="experiment",
)
ax.axhline(y=1.0, color="gray", linestyle="--", label="ideal")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Expectation value")
ax.legend(loc="upper left")

plt.show()

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/eef38976-0ca2-429a-b2dc-41aac69605f7-0.avif" alt="Output of the previous code cell" />

### Pasul 3: Execută folosind primitivele Qiskit
Acum ești gata să rulezi circuitul folosind primitiva Estimator.

Aici vei trimite cinci joburi separate, începând fără nicio suprimare sau atenuare a erorilor, și activând succesiv diverse opțiuni de suprimare și atenuare a erorilor disponibile în Qiskit Runtime. Pentru informații despre opțiuni, consultă următoarele pagini:

- [Prezentare generală a tuturor opțiunilor](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options)
- [Decuplare dinamică](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options)
- [Reziliență, inclusiv atenuarea erorilor de măsurare și extrapolarea la zgomot zero (ZNE)](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2)
- [Twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)

Deoarece aceste joburi pot rula independent unele de altele, poți folosi [modul batch](/guides/run-jobs-batch) pentru a permite Qiskit Runtime să optimizeze momentul execuției lor.

In [17]:
n_qubits = 50
reps = 1

# Construct circuit and observable
circuit = efficient_su2(n_qubits, entanglement="pairwise", reps=reps)
observable = SparsePauliOp.from_sparse_list(
    [("Z", [-1], 1.0)], num_qubits=n_qubits
)

# Assign parameters to circuit
params = rng.uniform(-np.pi, np.pi, size=circuit.num_parameters)
assigned_circuit = circuit.assign_parameters(params)
assigned_circuit.barrier()

# Construct mirror circuit
mirror_circuit = unitary_overlap(assigned_circuit, assigned_circuit)

# Transpile circuit and observable
isa_circuit = pass_manager.run(mirror_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Run jobs
pub = (isa_circuit, isa_observable)

jobs = []

with Batch(backend=backend) as batch:
    estimator = Estimator(mode=batch)
    estimator.options.environment.job_tags = [
        "TUT_CEM_LS"
    ]  # add tag for this large scale job
    # Set number of shots
    estimator.options.default_shots = 100_000
    # Disable runtime compilation and error mitigation
    estimator.options.resilience_level = 0

    # Run job with no error mitigation
    job0 = estimator.run([pub])
    jobs.append(job0)

    # Add dynamical decoupling (DD)
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XpXm"
    job1 = estimator.run([pub])
    jobs.append(job1)

    # Add readout error mitigation (DD + TREX)
    estimator.options.resilience.measure_mitigation = True
    job2 = estimator.run([pub])
    jobs.append(job2)

    # Add gate twirling (DD + TREX + Gate Twirling)
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    job3 = estimator.run([pub])
    jobs.append(job3)

    # Add zero-noise extrapolation (DD + TREX + Gate Twirling + ZNE)
    estimator.options.resilience.zne_mitigation = True
    estimator.options.resilience.zne.noise_factors = (1, 3, 5)
    estimator.options.resilience.zne.extrapolator = ("exponential", "linear")
    job4 = estimator.run([pub])
    jobs.append(job4)

# Retrieve the job results
results = [job.result() for job in jobs]

# Unpack the PUB results (there's only one PUB result in each job result)
pub_results = [result[0] for result in results]

# Unpack the expectation values and standard errors
expectation_vals = np.array(
    [float(pub_result.data.evs) for pub_result in pub_results]
)
standard_errors = np.array(
    [float(pub_result.data.stds) for pub_result in pub_results]
)

# Plot the expectation values
fig, ax = plt.subplots()
labels = ["No mitigation", "+ DD", "+ TREX", "+ Twirling", "+ ZNE"]
ax.bar(
    range(len(labels)),
    expectation_vals,
    yerr=standard_errors,
    label="experiment",
)
ax.axhline(y=1.0, color="gray", linestyle="--", label="ideal")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Expectation value")
ax.legend(loc="upper left")

plt.show()

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/d7d8408b-faf1-4eda-ab9c-bdeaab01ff53-0.avif" alt="Output of the previous code cell" />

### Pasul 4: Post-procesează și returnează rezultatul în formatul clasic dorit
În final, poți analiza datele. Aici vei prelua rezultatele joburilor, vei extrage valorile de așteptare măsurate din acestea și vei reprezenta grafic valorile, inclusiv barele de eroare de o deviație standard.